In [1]:
# Cell 1: import library
import os
import numpy as np
import pandas as pd
from dotenv import load_dotenv
from urllib.parse import quote_plus
from sqlalchemy import create_engine, text
from pathlib import Path

In [2]:
# Cell 2: connect to MySQL
PROJECT_ROOT = Path.cwd().parent
load_dotenv(PROJECT_ROOT / ".env")
user = os.getenv("MYSQL_USER")
pwd = quote_plus(os.getenv("MYSQL_PASSWORD"))
host = os.getenv("MYSQL_HOST")
port = os.getenv("MYSQL_PORT")
db = os.getenv("MYSQL_DB")
engine = create_engine(f"mysql+pymysql://{user}:{pwd}@{host}:{port}/{db}")
with engine.connect() as conn:
    print("Current database:", conn.execute(text("SELECT DATABASE()")).scalar())

Current database: olist_ecommerce


In [3]:
# Cell 3: headline numbers from the database, checked against notebooks 6-10
def one(sql):
    with engine.connect() as conn:
        return conn.execute(text(sql)).fetchone()

orders, customers, revenue = one("SELECT COUNT(*), COUNT(DISTINCT customer_unique_id), SUM(revenue) FROM valid_orders_t")
repeat = one("""SELECT COUNT(*) FROM (SELECT customer_unique_id FROM valid_orders_t
                GROUP BY customer_unique_id HAVING COUNT(*) > 1) t""")[0]
late_n, deliv_n = one("""SELECT SUM(order_delivered_customer_date > order_estimated_delivery_date), COUNT(*)
                         FROM valid_orders_t
                         WHERE order_status = 'delivered' AND order_delivered_customer_date IS NOT NULL""")
rv = pd.read_sql(text("""
    SELECT (v.order_delivered_customer_date > v.order_estimated_delivery_date) AS is_late,
           COUNT(*) AS reviewed_orders, AVG(r.review_score) AS avg_review,
           SUM(r.review_score <= 2) AS bad_reviews
    FROM valid_orders_t v JOIN latest_review_t r ON r.order_id = v.order_id
    WHERE v.order_status = 'delivered' AND v.order_delivered_customer_date IS NOT NULL
    GROUP BY is_late"""), engine).set_index("is_late")
rv = rv.astype(float)

orders, customers, repeat, late_n, deliv_n = int(orders), int(customers), int(repeat), int(late_n), int(deliv_n)
revenue = float(revenue)
bad_all = rv["bad_reviews"].sum()
checks = {
    "valid orders = 98198":              orders == 98198,
    "customers = 94982":                 customers == 94982,
    "revenue = 15738221.95":             round(revenue, 2) == 15738221.95,
    "repeat customers = 2887":           repeat == 2887,
    "delivered orders = 96469":          deliv_n == 96469,
    "late rate = 8.11%":                 round(late_n / deliv_n * 100, 2) == 8.11,
    "reviewed orders = 95823":           int(rv["reviewed_orders"].sum()) == 95823,
    "reviewed late = 7660":              int(rv.loc[1, "reviewed_orders"]) == 7660,
    "avg review on time / late = 4.29 / 2.57": (round(rv.loc[0, "avg_review"], 2), round(rv.loc[1, "avg_review"], 2)) == (4.29, 2.57),
    "bad reviews = 12271 (late 4141)":   (int(bad_all), int(rv.loc[1, "bad_reviews"])) == (12271, 4141),
}
for k, ok in checks.items():
    print("PASS" if ok else "FAIL", "-", k)

print(f"\nrepeat rate: {repeat / customers * 100:.2f}% of customers")
print(f"late orders: {late_n} of {deliv_n} ({late_n / deliv_n * 100:.2f}%)")
print(f"bad-review rate: on time {rv.loc[0, 'bad_reviews'] / rv.loc[0, 'reviewed_orders'] * 100:.2f}%, "
      f"late {rv.loc[1, 'bad_reviews'] / rv.loc[1, 'reviewed_orders'] * 100:.2f}%")

PASS - valid orders = 98198
PASS - customers = 94982
PASS - revenue = 15738221.95
PASS - repeat customers = 2887
PASS - delivered orders = 96469
PASS - late rate = 8.11%
PASS - reviewed orders = 95823
PASS - reviewed late = 7660
PASS - avg review on time / late = 4.29 / 2.57
PASS - bad reviews = 12271 (late 4141)

repeat rate: 3.04% of customers
late orders: 7825 of 96469 (8.11%)
bad-review rate: on time 9.22%, late 54.06%


In [4]:
# Cell 4: scenario: more repeat orders from one-time high spenders (spend in the top 40%, as in notebook 8)
cust = pd.read_sql(text("""SELECT customer_unique_id, COUNT(*) AS frequency, SUM(revenue) AS monetary
                           FROM valid_orders_t GROUP BY customer_unique_id"""), engine)
cust["monetary"] = cust["monetary"].astype(float)
cust = cust.sort_values("customer_unique_id").reset_index(drop=True)
cust["M"] = pd.qcut(cust["monetary"].rank(method="first"), 5, labels=[1, 2, 3, 4, 5]).astype(int)

hs = cust[(cust["frequency"] == 1) & (cust["M"] >= 4)]
print("PASS" if len(hs) == 35698 else "FAIL", "- one-time high spenders = 35698 (14648 recent + 21050 lapsed, notebook 8)")
print(f"their spend: R${hs['monetary'].sum():,.2f} ({hs['monetary'].sum() / revenue * 100:.1f}% of revenue), "
      f"average R${hs['monetary'].mean():.2f}")

rows = []
for p in [0.01, 0.02, 0.05]:
    extra = p * hs["monetary"].sum()       # assumption: each extra order is worth the customer's first order
    rows.append({"extra repeat rate": f"{p * 100:.0f}%", "extra customers": round(p * len(hs)),
                 "extra revenue (R$)": round(extra), "% of current revenue": round(extra / revenue * 100, 2)})
print("\nAssumption: a customer who is won back places ONE more order worth the same as the first. Illustration only.")
display(pd.DataFrame(rows))

PASS - one-time high spenders = 35698 (14648 recent + 21050 lapsed, notebook 8)
their spend: R$10,744,532.74 (68.3% of revenue), average R$300.98

Assumption: a customer who is won back places ONE more order worth the same as the first. Illustration only.


,extra repeat rate,extra customers,extra revenue (R$),% of current revenue
0,1%,357,107445,0.68
1,2%,714,214891,1.37
2,5%,1785,537227,3.41


In [5]:
# Cell 5: scenario: fewer late deliveries and the bad reviews linked to them (upper bound)
n_late, n_ok = rv.loc[1, "reviewed_orders"], rv.loc[0, "reviewed_orders"]
bad_late = rv.loc[1, "bad_reviews"] / n_late
bad_ok = rv.loc[0, "bad_reviews"] / n_ok
print(f"bad-review rate: late {bad_late * 100:.2f}% vs on time {bad_ok * 100:.2f}%")

rows = []
for cut in [0.10, 0.25, 0.50]:
    converted = cut * n_late                         # late orders that become on-time
    avoided = converted * (bad_late - bad_ok)        # assumes the whole gap is caused by lateness
    rows.append({"late orders reduced by": f"{cut * 100:.0f}%", "late reviewed orders avoided": round(converted),
                 "bad reviews avoided (upper bound)": round(avoided),
                 "% of all bad reviews": round(avoided / bad_all * 100, 1)})
print("\nAssumption: an order that stops being late gets the on-time bad-review rate. This is an upper bound, "
      "because late orders may also differ in other ways (notebook 10 found the odds ratio of about 12 survives controls).")
display(pd.DataFrame(rows))

bad-review rate: late 54.06% vs on time 9.22%

Assumption: an order that stops being late gets the on-time bad-review rate. This is an upper bound, because late orders may also differ in other ways (notebook 10 found the odds ratio of about 12 survives controls).


,late orders reduced by,late reviewed orders avoided,bad reviews avoided (upper bound),% of all bad reviews
0,10%,766,343,2.8
1,25%,1915,859,7.0
2,50%,3830,1717,14.0


## Business recommendations

All figures come from notebooks 6 to 11 and were re-checked against the database in Cell 3. The scenarios in Cells 4 and 5 are illustrations with the assumptions stated, not forecasts.

### Key facts
- 98,198 valid orders, 94,982 customers, R$15,738,221.95 in revenue. 96.96% of customers bought only once.
- 8.11% of delivered orders were late. Average review: 4.29 on time, 2.57 late. Bad reviews (score 1-2): 9.22% of on-time orders, 54.06% of late orders.
- Late orders are 8.0% of reviewed orders but 33.7% of all bad reviews. At most about 28% of bad reviews can be linked to lateness, and **66.3% come from on-time orders**.
- 35,698 customers who ordered once and spent in the top 40% account for 68.3% of revenue (R$10,744,532.74).

### 1. Bring back one-time high spenders (retention)
- **Evidence (notebooks 6, 8, 9):** only 3.04% of customers ordered again. The 35,698 one-time high spenders average R$300.98 (14,648 ordered recently, about 97 days before the end of the data; 21,050 are lapsed, about 342 days). The K-Means high-spender cluster is 12.8% of customers and 42.0% of revenue. Repeat buyers are not individually richer (average R$299 to R$322 against R$300 to R$303), so the value is in getting a second order.
- **Action:** pilot a follow-up to recent high spenders first, then to lapsed ones, with a random control group that receives nothing.
- **Scenario (assumption: each won-back customer places one more order equal to their first):** 1%, 2% or 5% of the group ordering again would add about R$107,000, R$215,000 or R$537,000 (0.68%, 1.37% or 3.41% of current revenue).
- **Measure:** share ordering again within 90 and 180 days, treated against control.
- **Limits:** no data on cost or response rates. The data ends in September 2018, so "lapsed" customers may not be lost.

### 2. Cut late deliveries where the time is lost: the carrier stage, and escalate early
- **Evidence (notebook 10):** late orders take 20.60 days longer than on-time ones (31.53 against 10.93). **17.77 of those days (86%) are in the carrier-to-customer stage;** handling adds 2.71 days and approval 0.11. Reviews fall sharply once an order is more than 2 days late: average 3.92 at 0 to 2 days late, 2.81 at 2 to 5 days (46.19% bad reviews), 1.89 at 5 to 10 days (73.89%).
- **Action:** (a) monitor transit time by carrier and route and escalate slow ones; (b) escalate orders that are about to pass 2 days late, before the review collapse; (c) test proactive delay messages to customers.
- **Scenario (upper bound: an order that stops being late gets the on-time bad-review rate):** cutting late orders by 10%, 25% or 50% would avoid about 343, 859 or 1,717 bad reviews (2.8%, 7.0% or 14.0% of all bad reviews).
- **Measure:** late rate and share of orders more than 5 days late by route; bad-review rate.
- **Limits:** the data does not separate carrier, distance and region as causes of slow transit. The effect of messages to customers is untested.

### 3. Give customers realistic delivery dates, by route
- **Evidence (notebooks 6, 7, 11):** the platform estimate averages 23.74 days against 12.56 actual. On the test period it was about 12 days too long (MAE 12.67 days), while the regression model's error was 3.72 days. Late rates by customer state differ: AL 23.93%, MA 19.67%, PI 15.97%, CE 15.32%, SE 15.22% and BA 14.04%, against 5.89% for SP. The odds of a late delivery are 1.72 times higher outside SP (small effect size, Cramer's V 0.13).
- **Action:** pilot delivery date ranges built from the model. Keep the model for date estimates only. The late-or-not classifier (AUROC 0.728, AP 0.132) is not reliable for alerts.
- **Measure:** estimate error in days, and customer reviews, with and without the model estimate.
- **Limits:** the late flag is measured against the estimate, so changing estimates changes the late rate. Judge the pilot with actual delivery days as well. Across three earlier validation periods the model was only 4% better than a simple line (the test gain was 29%), and only delivered orders are in the data.

### 4. Monitor sellers with a monthly scorecard
- **Evidence (notebooks 6, 10):** 23 of 627 sellers with 30+ delivered orders (3.7%) have a late rate above 20%; the average is 7.66%. The top 10% of sellers by late orders produce 72.7% of late orders but also handle 63.3% of all orders, so lateness is concentrated mostly because of volume. Late orders also spend longer in handling (5.34 days against 2.63).
- **Action:** a monthly scorecard (late rate, handling time, review score). Review the 23 sellers first.
- **Limits:** only about 13% of the extra time in late orders is in handling, so sellers can fix only a small part of the problem. Orders with several items use the highest-priced item's seller.

### 5. Look at the bad reviews that are not caused by lateness
- **Evidence (notebooks 6, 10):** 8,130 of 12,271 bad reviews (66.3%) come from on-time orders. office_furniture has the lowest average review (3.49 in notebook 6) with a late rate near the average (9.19% in notebook 10); bed_bath_table averages 3.90 on 9,295 orders. In the regression each extra item in an order goes with 61% higher odds of a bad review (odds ratio 1.61); 9,634 orders have more than one item and 1,275 have more than one seller.
- **Action:** read the review comments of the lowest-scoring categories (not analyzed in these notebooks) and check how multi-item and multi-seller orders are shipped.
- **Limits:** the causes are not tested. These are the places to look first.

### 6. Prepare for peak months
- **Evidence (notebook 6):** the monthly late rate reached 14.31% in Nov 2017, 15.99% in Feb 2018 and 21.36% in Mar 2018, against 1.36% in Jun 2018. February 2018 had fewer orders (6,624) than January (7,187), so volume alone does not explain it.
- **Action:** a weekly early-warning on late rate and transit time, and an investigation of the Feb to Mar 2018 episode (carrier, region).
- **Limits:** the cause is unknown.

## Priorities and measurement plan

### Priority order
| Priority | Recommendation | Why it comes first |
|---|---|---|
| 1 | 2. Carrier stage and early escalation | Late orders are linked to 33.7% of bad reviews, and 86% of their extra time is in transit. The largest identified driver of low reviews. |
| 2 | 1. Retention pilot on recent high spenders | The largest revenue pool (68.3% of revenue) and a cheap, testable pilot with a control group. |
| 3 | 3. Model-based delivery date ranges | The model's error (3.72 days) is much lower than the platform's (12.67). Pilot first, because the gain varies by period. |
| 4 | 5. Causes of non-late bad reviews | 66.3% of bad reviews are not about lateness, and the causes are not yet known. |
| 5 | 4. Seller scorecard and 6. Peak-month early warning | Useful monitoring, with smaller expected effect. |

### How to measure
- **Tests, not assumptions:** run recommendations 1, 2c and 3 as randomized pilots with a control group, and decide from the control comparison.
- **Metrics to track:** late rate and share of orders more than 5 days late (overall and by route), bad-review rate, share of customers ordering again within 90 and 180 days, and the error of the delivery estimate in days.
- **Re-run the checks:** repeat the reconciliation lines of these notebooks each quarter, and re-evaluate the model on the latest months before relying on it.

### Limits of this analysis
- The data is observational (Oct 2016 to Aug 2018, delivered orders only). It shows associations, not proven effects.
- There is no cost, margin, marketing or carrier data, so the scenarios show size, not profit.
- Scenario figures rest on stated assumptions and are illustrations. The late-delivery scenario is an upper bound.
- Review results use only orders that were reviewed.